# 6.1 字元切詞有什麼成本？


[1.1](https://birdhackor.github.io/tiny-perceptron-vlm/1.1.html) 每個字給一個ID，很容易理解「貓看狗」需要三格。但英文的 `perceptron` 若一字母一格，會有十格；中文若改成逐byte，又常一字需要三格。同一份原文用不同單位表示，模型所見的序列長度就不同，成本也跟着變。選擇怎樣拆文字，不只是換一張字表。

token是模型一次處理的文字片段，tokenizer是把原文拆成這些片段、轉成ID並能解迴文字的工具。「切詞」是常用翻譯，卻不保證片段正好是人類的詞；一個token可以是一字、多個字，或一個字元所需bytes中的部分，例如🙂的四個bytes分別當四個token。此處不是再把一個byte拆成位元；一byte一token時，每個byte仍完整包含八個位元。先分清原文、拆分單位與ID，後面改拆法纔不會把各種長短當同一個數字。

Unicode給文字元素統一編號，Python的 `len(字串)` 數這些碼點；一個碼點可以理解為Unicode中的一個文字編號，不一定等於肉眼的一個完整字符，因為重音或組合emoji可能用多個碼點。UTF-8則把文字存成0到255的bytes，每byte八位元，中文與emoji通常要多個byte。下面只用簡單單碼點例子。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
texts = ["小小貓", "cat", "🙂"]
for text in texts:
    raw = text.encode("utf-8")
    print(text, "Unicode碼點", len(text), "UTF-8 bytes", len(raw))

輸入三段文字，`.encode("utf-8")` 把字串轉為這種儲存編碼的byte序列。三行應分別看到3與9、3與3、1與4。兩種三字文本，實際存儲bytes不同；🙂顯示一個圖案卻需四byte。一字元一token時前兩段都是三格，一byte一token時則九格與三格，這些單位不能互換。

為何模型成本受影響？[3.6](https://birdhackor.github.io/tiny-perceptron-vlm/3.6.html) 的注意力權重有T×T格，T是token序列長度；從三格變九格，這張表便從9格變81格。字表則影響 [2.1](https://birdhackor.github.io/tiny-perceptron-vlm/2.1.html) 的V×D參數，V是可用token種類數。較細拆法常讓V較小、T較長，較粗拆法相反，不能只看字表小就說全模型省。

逐字元表還要處理未見字；固定256種byte則涵蓋任何UTF-8原文，能把沒見過的字拆得更細表示。這帶來可靠還原，也可能讓中文序列較長。下一節用常見相鄰片段合併，讓頻繁組合用更少格。

練習只加入 `"cat🙂"`，先預測Unicode碼點4、UTF-8 bytes7，再執行核對。再把英文cat換成貓，應是2與7：碼點減少，byte數卻一樣。你可以看到「幾字」不能直接當成「模型處理幾token」，還得說明採用哪種拆法。
